# Clases con Clase

Hasta ahora miramos muchas cosas: analizamos las variables, el target, la performance general de los modelos. Pero no profundizamos tanto en el estudio de las salidas (*outputs*) del modelo en sí, salvo la excepción de la curva ROC.

Antes de empezar, carguemos el entorno de trabajo.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px


from scipy.stats import gaussian_kde
from scipy.stats import ks_2samp

from sklearn.model_selection import StratifiedShuffleSplit

import lightgbm as lgb

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour



In [ ]:
base_path = ''
dataset_path = base_path + 'data/'
modelos_path = base_path + 'modelos/'
db_path = base_path + 'db/'
dataset_file = 'competencia_01.parquet.gzip'

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

mes_train = 202103
mes_test = 202105

# agregue sus semillas
semilla = 17
np.random.seed(semilla)
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

data = pd.read_parquet(dataset_path + dataset_file)


## Gradient boosting decision trees

Comenzaremos usando **LightGBM**. Si querés entender cómo funciona, se recomienda primero entender el algoritmo en el que se basa, **XGBoost**. Para una introducción completa, puedes consultar la [documentación de XGBoost](https://xgboost.readthedocs.io/en/stable/tutorials/model.html).

Aunque en la cátedra no somos grandes seguidores de Josh Starmer y su canal *StatQuest*, reconozco que sus series sobre *Gradient Boosting* y *XGBoost* son excelentes recursos. Aquí te dejamos los enlaces a esas dos series que realmente valen la pena:

[Serie Gradient Boosting](https://www.youtube.com/watch?v=3CC4N4z3GJc&list=PLblh5JKOoLUJjeXUvUE0maghNuY2_5fY6)

[Serie XGBoost](https://www.youtube.com/watch?v=OtD8wVaFm6E&list=PLblh5JKOoLULU0irPgs1SnKO6wqVjKUsQ)

Finalmente, analizaremos las diferencias clave que ofrece **LightGBM** frente a XGBoost. Puedes explorar más sobre ello en la [documentación de features de LightGBM](https://lightgbm.readthedocs.io/en/stable/Features.html).

No olvides tener a mano la [documentación de LightGBM](https://lightgbm.readthedocs.io/) y la [lista completa de sus parámetros](https://lightgbm.readthedocs.io/en/latest/Parameters.html).

Este es un algoritmo muy usado en el mercado, recomiendo dedicarle el tiempo necesario para aprenderlo bien.

A su vez, trabajaremos con modelos binarios, para lo que consolidaremos las clases **CONTINUA** con **BAJA+1**.

In [ ]:
data['clase_binaria'] = np.where(data['clase_ternaria'] == 'BAJA+2', 1, 0)

Seguimos trabajando con los meses de **Marzo** y **Mayo** y serializamos al formato de **LGBM**

In [ ]:
train_data = data[data['foto_mes'] == mes_train]
test_data = data[data['foto_mes'] == mes_test]

X_train = train_data.drop(['clase_ternaria', 'clase_binaria'], axis=1)
y_train_binaria = train_data['clase_binaria']

X_test = test_data.drop(['clase_ternaria', 'clase_binaria'], axis=1)
y_test_binaria = test_data['clase_binaria']
y_test_class = test_data['clase_ternaria']


train_data_lgb = lgb.Dataset(X_train, label=y_train_binaria)
test_data_lgb = lgb.Dataset(X_test, label=y_test_binaria)

**LGBM** tiene muchas comodidades a la hora de trabajar con optimización, entre ellas:
- Posibilidad de utilizar métricas propias.
- Utilizar **cv** de forma versátil.
- Una larga lista de parámetros para optimizar: [Learning Control Parameters](https://lightgbm.readthedocs.io/en/latest/Parameters.html#learning-control-parameters).

A continuación, procederemos a optimizar **LightGBM** utilizando la librería **Optuna**. Cabe destacar que las optimizaciones que realizaremos son básicas y están diseñadas para ejecutarse en pocos minutos. Será su responsabilidad ampliar tanto el rango de búsqueda como el tiempo de optimización para obtener un modelo más competitivo.


In [ ]:
sss_opt = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=semillas[0])

def ganancia_prob_lgm(y_pred, data, prop=0.3):
  y_true = data.get_label()
  ganancia = np.where(y_true == 1, ganancia_acierto, 0) - np.where(y_true == 0, costo_estimulo, 0)
  return 'gan_prob', ganancia[y_pred >= 0.025].sum() / prop, True



def objective(trial):

    num_leaves = trial.suggest_int('num_leaves', 8, 100)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3) # mas bajo, más iteraciones necesita
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 1, 1000)
    feature_fraction = trial.suggest_float('feature_fraction', 0.1, 1.0)
    bagging_fraction = trial.suggest_float('bagging_fraction', 0.1, 1.0)

    params = {
        'objective': 'binary',
        'metric': 'custom',
        'boosting_type': 'gbdt',
        'first_metric_only': True,
        'boost_from_average': True,
        'feature_pre_filter': False,
        'max_bin': 31,
        'num_leaves': num_leaves,
        'learning_rate': learning_rate,
        'min_data_in_leaf': min_data_in_leaf,
        'feature_fraction': feature_fraction,
        'bagging_fraction': bagging_fraction,
        'seed': semillas[0],
        'verbose': -1
    }

    cv_results = lgb.cv(
        params,
        train_data_lgb,
        num_boost_round=1000, # modificar, subir y subir...
        callbacks=[lgb.early_stopping(stopping_rounds=int(50 + 5 / learning_rate), verbose=False)],
        feval=ganancia_prob_lgm,
        folds=sss_opt.split(X_train, y_train_binaria), # usamos StratifiedShuffleSplit en vez del k-fold interno
        seed=semillas[0]
    )

    max_gan = max(cv_results['valid gan_prob-mean'])
    best_iter = cv_results['valid gan_prob-mean'].index(max_gan) + 1

    # Guardamos cual es la mejor iteración del modelo
    trial.set_user_attr("best_iter", best_iter)

    return max_gan


storage_name = "sqlite:///" + db_path + "optimization_lgbm.db"
study_name = "exp_301_lgbm_sss"

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)


In [ ]:
# study.optimize(objective, n_trials=250)

Analicemos los resultados de la optimización, como venimos haciendo en clases anteriores.


In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
plot_param_importances(study)

El **learning rate** es un parámetro que tiene que ir acompañado por más árboles.

In [ ]:
plot_slice(study)

In [ ]:
plot_contour(study)

In [ ]:
plot_contour(study, params=['learning_rate','min_data_in_leaf'] )

Y finalmente tomamos el mejor modelo y lo entrenamos con la totalidad de los datos

In [ ]:
best_iter = study.best_trial.user_attrs["best_iter"]
print(f"Mejor cantidad de árboles para el mejor model {best_iter}")
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'first_metric_only': True,
    'boost_from_average': True,
    'feature_pre_filter': False,
    'max_bin': 31,
    'num_leaves': study.best_trial.params['num_leaves'],
    'learning_rate': study.best_trial.params['learning_rate'],
    'min_data_in_leaf': study.best_trial.params['min_data_in_leaf'],
    'feature_fraction': study.best_trial.params['feature_fraction'],
    'bagging_fraction': study.best_trial.params['bagging_fraction'],
    'seed': semillas[0],
    'verbose': 0
}

model = lgb.train(params,
                  train_data_lgb,
                  num_boost_round= 250) #best_iter)


Observamos las variables más importantes para el modelo:

In [ ]:
importances = model.feature_importance()
feature_names = X_train.columns.tolist()
importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False)
importance_df[importance_df['importance'] > 0]


Para guardar el modelo para poder utilizarlo más adelante, no es necesario guardarlo como *pickle*, la librería nos permite guardarlo en formato texto

In [ ]:
# model.save_model(modelos_path + 'lgb_first.txt')

Y recuperar el mismo desde ese formato

In [ ]:
# model = lgb.Booster(model_file=modelos_path + 'lgb_first.txt')

Y sobre ambos modelos obtenemos la predicción de **Mayo**

In [ ]:
y_pred_lgm = model.predict(X_test)

Finalmente medimos las ganancias en **Mayo**

In [ ]:
def gan_prob(y_pred, y_true, prop = 1):
  ganancia = np.where(y_true == 1, ganancia_acierto, 0) - np.where(y_true == 0, costo_estimulo, 0)
  return ganancia[y_pred >= 0.025].sum() / prop

print("Ganancia LGBM en Mayo:", gan_prob(y_pred_lgm, y_test_binaria))


Haremos de nuevo un análisis que empezamos a ver varias clases atrás, de no quedarnos con un solo modelo de la optimización. Veamos los 3 mejores

In [ ]:
top_n = 3
top_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value,
    reverse=True
)[:top_n]

train_data = lgb.Dataset(X_train, label=y_train_binaria)

resultados = []

for i, trial in enumerate(top_trials, start=1):
    # best_iter = trial.user_attrs["best_iter"]

    params = {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'first_metric_only': True,
        'boost_from_average': True,
        'feature_pre_filter': False,
        'max_bin': 31,
        'num_leaves': trial.params['num_leaves'],
        'learning_rate': trial.params['learning_rate'],
        'min_data_in_leaf': trial.params['min_data_in_leaf'],
        'feature_fraction': trial.params['feature_fraction'],
        'bagging_fraction': trial.params['bagging_fraction'],
        'seed': semillas[0],
        'verbose': -1
    }

    model = lgb.train(params, train_data_lgb, num_boost_round=250) #, num_boost_round=best_iter)

    y_pred_mayo = model.predict(X_test)
    ganancia_mayo = gan_prob(y_pred_mayo, y_test_binaria)

    resultados.append({
        "orden": i,
        "trial": trial.number,
        "valor_cv": trial.value,
        "best_iter": 250, # best_iter,
        "ganancia_mayo": ganancia_mayo,
        **trial.params
    })

df_top3 = pd.DataFrame(resultados)
df_top3


¿Sigue siendo el mejor de la optimización el mejor en **Mayo**?

¿Y si todo esto tuvo que ver con la suerte de la semilla? Para descartarlo, entrenemos el mismo modelo con varias semillas distintas y comparemos los resultados.


In [ ]:
n_semillas = 20
train_data = lgb.Dataset(X_train, label=y_train_binaria)

rows = []

for _, fila in df_top3.iterrows():  # cambiar por df_top3 si es tu nombre
    for seed in semillas[:n_semillas]:
        params = {
            'objective': 'binary',
            'boosting_type': 'gbdt',
            'first_metric_only': True,
            'boost_from_average': True,
            'feature_pre_filter': False,
            'max_bin': 31,
            'num_leaves': int(fila['num_leaves']),
            'learning_rate': fila['learning_rate'],
            'min_data_in_leaf': int(fila['min_data_in_leaf']),
            'feature_fraction': fila['feature_fraction'],
            'bagging_fraction': fila['bagging_fraction'],
            'seed': seed,
            'verbose': -1
        }

        model = lgb.train(params, train_data, num_boost_round=250) #int(fila['best_iter']))

        y_pred_mayo = model.predict(X_test)
        ganancia_mayo = gan_prob(y_pred_mayo, y_test_binaria)

        rows.append({
            "trial": fila['trial'],
            "seed": seed,
            "ganancia_mayo": ganancia_mayo
        })

df_semillas = pd.DataFrame(rows)

estadisticas = (
    df_semillas.groupby('trial')['ganancia_mayo']
    .agg(['mean', 'std', 'min', 'max', 'median'])
    .sort_values('mean', ascending=False)
)
estadisticas


Grafiquemos la distribución de ganancias obtenidas para poder comparar los modelos entre sí.

In [ ]:
orden_trials = [str(t) for t in estadisticas.index]

fig = go.Figure()

for trial in estadisticas.index:
    datos_trial = df_semillas[df_semillas['trial'] == trial]
    fig.add_trace(go.Box(
        y=datos_trial['ganancia_mayo'],
        x=[str(trial)] * len(datos_trial),
        name=str(trial),
        boxpoints=False,
        showlegend=False
    ))

primera_semilla = df_semillas[df_semillas['seed'] == semillas[0]]
fig.add_trace(go.Scatter(
    x=[str(t) for t in primera_semilla['trial']],
    y=primera_semilla['ganancia_mayo'],
    mode='markers',
    marker=dict(color='red', size=12, line=dict(color='black', width=1)),
    name=f'semilla {semillas[0]}'
))

fig.update_layout(
    title=f"Ganancia en Mayo por modelo ({n_semillas} semillas)",
    xaxis_title="Trial (modelo)",
    yaxis_title="Ganancia en Mayo",
    xaxis=dict(categoryorder='array', categoryarray=orden_trials),
    template='plotly_white'
)

fig.show()


De los modelos entrenados, nos quedamos con la iteración que consideremos más adecuada, no necesariamente la de mayor ganancia puntual, sino la que ofrezca el mejor equilibrio entre ganancia y estabilidad.


In [ ]:
model_best_iter = 250

model_params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'first_metric_only': True,
    'boost_from_average': True,
    'feature_pre_filter': False,
    'max_bin': 31,
    'num_leaves': 16,
    'learning_rate': 0.019868,
    'min_data_in_leaf': 816,
    'feature_fraction': 0.277155,
    'bagging_fraction': 0.634130,
    'seed': semillas[0],
    'verbose': -1
}

my_model = lgb.train(
    model_params,
    train_data_lgb,
    num_boost_round=model_best_iter
)



In [ ]:
y_pred_my_model = my_model.predict(X_test)


## Más métricas

Pasaremos a estudiar la salida de este modelo en particular, empezando estudiando las probabilidades que entrega para **Mayo**

In [ ]:

grupos = [
    ('Distribución de la salida (todas las clases)', y_pred_my_model, 'gray'),
    ('Distribución de la salida - BAJA+2', y_pred_my_model[y_test_class.values == 'BAJA+2'], 'red'),
    ('Distribución de la salida - BAJA+1', y_pred_my_model[y_test_class.values == 'BAJA+1'], 'orange'),
    ('Distribución de la salida - CONTINUA', y_pred_my_model[y_test_class.values == 'CONTINUA'], 'blue'),
]

fig = make_subplots(rows=4, cols=1, shared_xaxes=True, subplot_titles=[g[0] for g in grupos])

for i, (titulo, datos, color) in enumerate(grupos, start=1):
    fig.add_trace(go.Histogram(
        x=datos, nbinsx=50, histnorm='probability density',
        marker_color=color, opacity=0.6, showlegend=False
    ), row=i, col=1)

    kde = gaussian_kde(datos)
    x_kde = np.linspace(datos.min(), datos.max(), 200)
    fig.add_trace(go.Scatter(
        x=x_kde, y=kde(x_kde), mode='lines',
        line=dict(color=color), showlegend=False
    ), row=i, col=1)

fig.update_layout(height=1000, width=700, template='plotly_white')
fig.update_xaxes(title_text='Probabilidad predicha', row=4, col=1)
fig.show()


Notemos que, en ningún caso, la probabilidad se acerca al 0.5 que uno esperaría en un dataset "balanceado".

### Test de Kolmogorov-Smirnov (KS)

El **KS** es una medida de qué tan bien separa un modelo dos poblaciones: en nuestro caso, los clientes que se dan de baja (clase 1) de los que continúan (clase 0). Se calcula comparando las funciones de distribución acumulada (CDF) de las probabilidades predichas para cada grupo, y el valor de KS es la **máxima distancia vertical** entre esas dos curvas.

- KS = 0 significa que ambas distribuciones son idénticas: el modelo no distingue nada entre clases.
- KS = 1 significa separación perfecta: no hay ningún solapamiento entre las probabilidades de ambas clases.

En la práctica, valores de KS entre 0.3 y 0.5 ya se consideran buenos para modelos de scoring, y valores mayores a 0.5 son raros salvo que haya algo de *leakage* de información.

El KS es una métrica clásica en la industria financiera y de riesgo crediticio (*credit scoring*), donde es prácticamente un estándar junto con el AUC. A diferencia del AUC, que resume la capacidad discriminativa del modelo en un solo número agregando todos los puntos de corte, el KS identifica **el punto de corte específico** donde la separación entre clases es máxima, lo cual muchas áreas de riesgo usan directamente como referencia para fijar el umbral de decisión del modelo.


In [ ]:
pred_pos = y_pred_my_model[y_test_binaria.values == 1]
pred_neg = y_pred_my_model[y_test_binaria.values == 0]

ks_stat, ks_pvalue = ks_2samp(pred_pos, pred_neg)

x_pos = np.sort(pred_pos)
cdf_pos = np.arange(1, len(x_pos) + 1) / len(x_pos)

x_neg = np.sort(pred_neg)
cdf_neg = np.arange(1, len(x_neg) + 1) / len(x_neg)

# punto donde se da la máxima distancia entre las dos CDF
todos = np.sort(np.concatenate([x_pos, x_neg]))
cdf_pos_interp = np.searchsorted(x_pos, todos, side='right') / len(x_pos)
cdf_neg_interp = np.searchsorted(x_neg, todos, side='right') / len(x_neg)
idx_max = np.argmax(np.abs(cdf_pos_interp - cdf_neg_interp))
x_ks = todos[idx_max]

fig = go.Figure()

fig.add_trace(go.Scatter(x=x_pos, y=cdf_pos, mode='lines', name='Clase 1 (Baja)', line=dict(color='red')))
fig.add_trace(go.Scatter(x=x_neg, y=cdf_neg, mode='lines', name='Clase 0 (Continúa)', line=dict(color='blue')))

fig.add_trace(go.Scatter(
    x=[x_ks, x_ks], y=[cdf_neg_interp[idx_max], cdf_pos_interp[idx_max]],
    mode='lines', name=f'KS = {ks_stat:.4f}',
    line=dict(color='black', dash='dash')
))

fig.update_layout(
    title='Test KS - distribución acumulada de probabilidades por clase',
    xaxis_title='Probabilidad predicha',
    yaxis_title='CDF',
    template='plotly_white'
)

fig.show()

print(f"KS = {ks_stat:.4f} (p-value = {ks_pvalue:.4g})")


### Ganancia acumulada

La **ganancia acumulada** ordena a todos los clientes según la probabilidad que el modelo les asignó, de mayor a menor, y va sumando la ganancia real a medida que se los va "estimulando" en ese orden: `+ganancia_acierto` cuando el cliente es realmente positivo (se da de baja), y `-costo_estimulo` en cualquier otro caso. El resultado es una curva que crece mientras los aciertos superan a los costos, llega a un máximo, y después empieza a bajar porque se sigue gastando en estimular clientes sin ganar nada a cambio.

- El **punto más alto de la curva** es la ganancia máxima posible con este modelo, y la cantidad de clientes en ese punto es la **cantidad óptima de envíos**.
- La probabilidad marcada en el gráfico (`p`) es el **umbral de corte** que corresponde a ese óptimo: el punto de corte ideal, según los datos de test, para decidir a quién estimular.
- A diferencia del AUC o el KS, que son medidas de calidad general del modelo, la ganancia acumulada está expresada directamente en las **unidades de negocio** (plata), por lo que es la métrica que más importa de cara a la competencia: no alcanza con discriminar bien, hay que hacerlo en el rango de corte que efectivamente maximiza el resultado económico.

Es el enfoque estándar en cualquier problema donde exista un **costo por acción** (contactar, estimular, revisar) y un **premio condicional** a acertar: campañas de retención, cobranzas, detección de fraude, marketing directo. En todos estos casos, la pregunta relevante no es solo "¿el modelo discrimina bien?" sino "¿hasta qué punto conviene actuar, dado el costo de cada intervención y el beneficio de cada acierto?" — algo que el AUC no responde por sí solo, pero que la ganancia acumulada muestra de forma directa.


In [ ]:
import plotly.graph_objects as go

orden = np.argsort(y_pred_my_model)[::-1]
ganancia_ordenada = np.where(y_test_binaria.values == 1, ganancia_acierto, -costo_estimulo)[orden]
ganancia_acumulada = np.cumsum(ganancia_ordenada)

mejor_corte = np.argmax(ganancia_acumulada) + 1
mejor_ganancia = ganancia_acumulada[mejor_corte - 1]
p_corte = y_pred_my_model[orden][mejor_corte - 1]

limite = 20000

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, limite + 1)), y=ganancia_acumulada[:limite],
    mode='lines', name='Ganancia acumulada', line=dict(color='blue')
))

fig.add_vline(
    x=mejor_corte, line_dash='dash', line_color='black',
    annotation_text=f'Corte óptimo: {mejor_corte} clientes',
    annotation_position='top'
)

fig.add_trace(go.Scatter(
    x=[mejor_corte], y=[mejor_ganancia],
    mode='markers+text', marker=dict(color='red', size=10),
    text=[f'p = {p_corte:.4f}'], textposition='middle right',
    showlegend=False
))

fig.update_layout(
    title='Ganancia acumulada según cantidad de clientes estimulados',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

print(f"Máxima ganancia: {mejor_ganancia:.2f} en el corte {mejor_corte}, con probabilidad de corte p = {p_corte:.4f}")



### Lift

El **Lift** mide cuánto mejor es el modelo respecto de elegir clientes al azar, para una cantidad dada de clientes contactados. Se calcula como el cociente entre la tasa de positivos reales dentro del grupo seleccionado (los top-N según el modelo) y la tasa de positivos en toda la población:

$$\text{Lift}(N) = \frac{\text{tasa de positivos en el top-N}}{\text{tasa de positivos en la población total}}$$

- Lift = 1 significa que el modelo no aporta nada: seleccionar esos N clientes es equivalente a elegirlos al azar.
- Lift = 3, por ejemplo, significa que en ese grupo hay 3 veces más positivos de los que se esperarían por azar.
- El lift siempre es más alto en los primeros cortes (los clientes con mayor probabilidad) y tiende a 1 a medida que el corte crece hasta cubrir toda la población.

Es una métrica muy usada en **marketing directo y campañas comerciales**, porque responde una pregunta muy concreta para el negocio: *"si solo puedo contactar a N clientes, ¿cuánto mejor me va usando el modelo en vez de elegir al azar?"*. 

El lift es una medida relativa y adimensional, más fácil de comunicar a áreas de negocio que no manejan los costos exactos del modelo, y más comparable entre campañas o modelos distintos.


In [ ]:
orden = np.argsort(y_pred_my_model)[::-1]
y_true_ordenado = y_test_binaria.values[orden]

cortes = np.arange(1000, 20001, 1000)
tasa_base = y_test_binaria.values.mean()

lift_valores = [y_true_ordenado[:corte].mean() / tasa_base for corte in cortes]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cortes, y=lift_valores,
    mode='lines+markers', marker=dict(color='green'), line=dict(color='green'),
    name='Lift'
))

fig.add_hline(y=1, line_dash='dash', line_color='gray', annotation_text='Azar (lift = 1)')

fig.update_layout(
    title='Lift según cantidad de clientes estimulados',
    xaxis_title='Cantidad de clientes (ordenados por probabilidad descendente)',
    yaxis_title='Lift',
    xaxis=dict(tickvals=cortes, tickangle=-45),
    template='plotly_white'
)

fig.show()


### Captura

Mide, para la misma cantidad de clientes contactados, qué proporción de los **BAJA+2 reales** (los que de verdad se quieren atrapar) el modelo va "capturando" a medida que avanza en la lista ordenada por probabilidad. Va de 0 a 1: si llegás al final de la lista, capturaste el 100% de los BAJA+2 porque contactaste a todo el mundo.

In [ ]:


orden = np.argsort(y_pred_my_model)[::-1]
y_true_ordenado = y_test_binaria.values[orden]

total_positivos = y_true_ordenado.sum()
captura = np.cumsum(y_true_ordenado) / total_positivos

n = len(y_true_ordenado)
percentiles = np.arange(1, n + 1) / n * 100

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=percentiles, y=captura,
    mode='lines', name='Modelo',
    line=dict(color='blue')
))

fig.update_layout(
    title='Captura acumulada de BAJA+2 - modelo trial 99',
    xaxis_title='Percentil de clientes (ordenados por probabilidad descendente)',
    yaxis_title='Proporción de BAJA+2 capturados',
    xaxis_range=[0, 100],
    template='plotly_white'
)

fig.show()


## (anti)Discriminación

Hasta ahora diferenciábamos las clases **BAJA+1** y **BAJA+2**, ya que tienen pesos distintos en la matriz de ganancia. Sin embargo, en todos los análisis realizados observamos que su comportamiento es bastante similar. ¿Nos convendría entonces romper ese esquema y fusionarlas en una sola clase? Probemos usando los mismos parámetros del mejor modelo optimizado para solo **BAJA+2**, lo que implica que no necesariamente es el mejor para esta nueva clase.


In [ ]:
data['clase_binaria2'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)

train_data2 = data[data['foto_mes'] == mes_train]
test_data2 = data[data['foto_mes'] == mes_test]

cols_a_descartar = ['clase_ternaria', 'clase_binaria', 'clase_binaria2']

X_train2 = train_data2.drop(cols_a_descartar, axis=1)
y_train_binaria2 = train_data2['clase_binaria2']

X_test2 = test_data2.drop(cols_a_descartar, axis=1)
y_test_binaria2 = test_data2['clase_binaria2']
y_test_class2 = test_data2['clase_ternaria']

train_data_lgb2 = lgb.Dataset(X_train2, label=y_train_binaria2)

model_baja1_2 = lgb.train(
    model_params,
    train_data_lgb2,
    num_boost_round=250
)

y_pred_baja1_2 = model_baja1_2.predict(X_test2)

def ganancia_real(y_pred, y_true_class, umbral=0.025):
    ganancia = np.where(y_true_class == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return ganancia[y_pred >= umbral].sum()

ganancia_nueva = ganancia_real(y_pred_baja1_2, y_test_class2.values)
print(f"Ganancia en Mayo (target BAJA+1+BAJA+2 vs CONTINUA): {ganancia_nueva:.2f}")


¡Chan! La ganancia bajó bastante. ¿No estaremos omitiendo algo? Volvamos a graficar la ganancia acumulada para revisarlo.


In [ ]:
orden2 = np.argsort(y_pred_baja1_2)[::-1]
clase_ordenada2 = y_test_class2.values[orden2]
ganancia_ordenada2 = np.where(clase_ordenada2 == 'BAJA+2', ganancia_acierto, -costo_estimulo)
ganancia_acumulada2 = np.cumsum(ganancia_ordenada2)

mejor_corte2 = np.argmax(ganancia_acumulada2) + 1
mejor_ganancia2 = ganancia_acumulada2[mejor_corte2 - 1]
p_corte2 = y_pred_baja1_2[orden2][mejor_corte2 - 1]

limite = 20000

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, limite + 1)), y=ganancia_acumulada2[:limite],
    mode='lines', name='Ganancia acumulada', line=dict(color='blue')
))

fig.add_vline(
    x=mejor_corte2, line_dash='dash', line_color='red',
    annotation_text=f'Corte óptimo: {mejor_corte2} clientes',
    annotation_position='top'
)

fig.add_trace(go.Scatter(
    x=[mejor_corte2], y=[mejor_ganancia2],
    mode='markers+text', marker=dict(color='red', size=10),
    text=[f'p = {p_corte2:.4f}'], textposition='middle right',
    showlegend=False
))

fig.update_layout(
    title='Ganancia acumulada - target BAJA+1+BAJA+2 vs CONTINUA',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

print(f"Máxima ganancia: {mejor_ganancia2:.2f} en el corte {mejor_corte2}, con probabilidad de corte p = {p_corte2:.4f}")


¡Cambió el punto de corte! Lo cual tiene todo el sentido, ya que cambia la proporción de las clases. Ahora podemos observar una gran mejora en la ganancia.

Y ahora, ¿cómo elegimos el punto de corte óptimo? Bueno, esa es una de las tareas más importantes del alumno. Una pista: cambiemos la forma de pensar y, en vez de mirar la probabilidad, miremos la cantidad de envíos. Probemos entonces con cortes de a 500 casos.


In [ ]:
corte_min = 4000
corte_max = 12000

paso = 500

cortes_tabla = np.arange(corte_min + paso, corte_max + 1, paso)
cortes_tabla = cortes_tabla[cortes_tabla <= len(ganancia_acumulada2)]

tabla_ganancia = pd.DataFrame({
    'corte': cortes_tabla,
    'ganancia_acumulada': ganancia_acumulada2[cortes_tabla - 1]
})

tabla_ganancia


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=tabla_ganancia['corte'], y=tabla_ganancia['ganancia_acumulada'],
    mode='lines+markers', line=dict(color='blue'), marker=dict(size=6)
))

fig.update_layout(
    title='Ganancia acumulada por corte',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()


Para confirmar que la diferencia observada no es producto del azar, repitamos el experimento con múltiples semillas y comparemos las distribuciones de ganancia.


In [ ]:
n_semillas = 20
cortes = np.arange(4000, 12001, 500)

rows = []

for seed in semillas[:n_semillas]:
    params_seed = dict(model_params)  # Copiamos los parámetros del mejor modelo
    params_seed['seed'] = seed

    model_seed = lgb.train(params_seed, train_data_lgb2, num_boost_round=250)
    y_pred_seed = model_seed.predict(X_test)

    orden_seed = np.argsort(y_pred_seed)[::-1]
    ganancia_ordenada_seed = np.where(y_test_binaria.values[orden_seed] == 1, ganancia_acierto, -costo_estimulo)
    ganancia_acumulada_seed = np.cumsum(ganancia_ordenada_seed)

    for corte in cortes:
        rows.append({
            "seed": seed,
            "corte": corte,
            "ganancia": ganancia_acumulada_seed[corte - 1]
        })

df_ganancia_cortes = pd.DataFrame(rows)

estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median'])
)
estadisticas_cortes


In [ ]:
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median',
          lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
)
estadisticas_cortes.columns = ['mean', 'std', 'min', 'max', 'median', 'q1', 'q3']

fig = go.Figure()

fig.add_trace(go.Box(
    x=[str(c) for c in estadisticas_cortes.index],
    q1=estadisticas_cortes['q1'],
    median=estadisticas_cortes['median'],
    q3=estadisticas_cortes['q3'],
    lowerfence=estadisticas_cortes['min'],
    upperfence=estadisticas_cortes['max'],
    mean=estadisticas_cortes['mean'],
    sd=estadisticas_cortes['std'],
    boxmean=True,
))

fig.update_layout(
    title=f'Distribución de ganancia por corte ({n_semillas} semillas)',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()


In [ ]:
fig = px.line(
    df_ganancia_cortes.sort_values(['seed', 'corte']),
    x='corte', y='ganancia', color='seed',
    markers=True,
    title='Ganancia por corte, una línea por semilla'
)

fig.update_layout(
    xaxis_title='Corte (cantidad de clientes)',
    yaxis_title='Ganancia',
    template='plotly_white'
)

fig.show()

## Tarea:

1. **Generar Dataset**
   - Utilice las técnicas de *feature engineering* vistas en las clases anteriores para generar un nuevo conjunto de datos.

2. **Optimización de LightGBM (LGBM)**
   - Ajuste el modelo de LightGBM utilizando una mayor cantidad de árboles y realice una exploración más exhaustiva de los hiperparámetros para mejorar su rendimiento.
   - Repita la optimización considerando los dos esquemas de clase binaria (**BAJA+2** vs. resto, y **BAJA+1**+**BAJA+2** vs. **CONTINUA**) y compare los resultados obtenidos con cada uno.
   - Revise la documentación de los parámetros de LightGBM. Evalúe la inclusión de otros parámetros en el proceso de optimización, y ajuste el modelo con estos nuevos parámetros.

3. **Selección del Mejor Modelo**
   - Entre los cinco mejores modelos obtenidos en cada optimización, seleccione el que considere más adecuado para la competencia en Kaggle.
   - Documente las pruebas que realizó para seleccionar el mejor modelo. Justifique su decisión con métricas relevantes y análisis comparativos.
   - Incorpore en este análisis la visión de un leaderboard público y privado, simulando cómo se comportaría el modelo elegido en ambos escenarios.

4. Escriba y comparta por **Zulip** el siguiente código:
   - ¿Qué métrica optimiza cuando tiene las dos bajas juntas?
